## Comp2-1 scaleTypedata

mainly designed for clinical data modelinganddepict。Typical application scenario explorationrad_scorefinal clinical diagnosis role。

## OnekeyStep

1. Data validation，check data formatiswhether correct。
3. view some statistical information，checkdatatime existinoutliers。
4. regularization，transform the data to follow N~(0, 1)。
5. Filter by correlation coefficient, e.g.,spearman、personand filter out features。
6. Construct train and test sets，Random split is used here, Normal multi-center validation, Two datasets tailored to the scenario are required。
7. throughLassoFilter features，Select all but the non-0items as features for subsequent modeling。
8. Use machine learning algorithms, e.g.,LR、SVM、RFetc.perform the task。
9. Model results visualization, e.g.,AUC、ROC curve，confusion matrix, etc.。


# filter clinical features

need to be adjusted to your case，Filter features，general filteringpvalue<0.05。

In [ ]:
import pandas as pd
import numpy as np
from collections import namedtuple
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
import onekey_algo.custom.components as okcomp
from onekey_algo.custom.components.comp1 import fillna
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300

## 1.、Data validation
first check the diagnostic data，If displayed`checkthrough！`the subsequent steps can run normally, otherwise, otherwisePlease adjust the data according to the prompts。

data fileinall dataisnumericType，orcanmap to numeric valuesType，Here the`label`in some cases mayisnon-numeric，need to customize the numeric function。

**Note：inusetree modeltime，canexistinmissing，butislinear models do not allow missing values，please fill the missing values yourself as needed**

In [ ]:
# read the data，Bsupra-diagnosis positive=1，bc_data.csvisneedread the data。
data_file = r'clinic_sel.csv'
labels = [get_param_in_cwd('task_column')]
featrues_not_use = ['ID']

structed_data = pd.read_csv(data_file, header=0)
structed_data

### feature dimension

In [ ]:
# deleteIDthis column。
ids = structed_data['ID']
structed_data = structed_data.drop(featrues_not_use, axis=1)
structed_data

## 2.、data statistics

1. count，count the number of samples。
2. mean、std, mean of the corresponding feature、variance
3. min, 25%, 50%, 75%, max，minimum of the corresponding feature, 25,50,75quantile, maximum。

In [ ]:
structed_data.describe()

## 3.、regularization

clinical featuresOptionalisnoregularization，defaultnotperform

`normalize_df` isonekeyregularized inAPI，transform data to 0mean1variance。Regularization method:

$column = \frac{column - mean}{std}$

In [ ]:
# from onekey_algo.custom.components.comp1 import normalize_df
# data = normalize_df(structed_data, not_norm=labels + ['group'])
# data = data.dropna(axis=1)
# data.describe()
data = structed_data
data

## 4.、Correlation coefficient

Methods for computing correlation coefficient:3options available
1. pearson （Pearson correlation coefficient）: standard correlation coefficient

2. kendall (Kendall correlation coefficient) : Kendall Tau correlation coefficient

3. spearman (Spearman correlation coefficient): Spearman rank correlation

Three correlation coefficients reference：https://blog.csdn.net/zmqsdu9001/article/details/82840332

In [ ]:
# if you need to selectCorrelation coefficientusecorrespondingCorrelation coefficienti.e.may。
# pearson_corr = data.corr('pearson')
# kendall_corr = data.corr('kendall')
spearman_corr = data.corr('spearman')

import seaborn as sns
import matplotlib.pyplot as plt
from onekey_algo.custom.components.comp1 import draw_matrix
plt.figure(figsize=(10.0, 8.0))

# Select the correlation coefficient to visualize
draw_matrix(spearman_corr, annot=True, cmap='YlGnBu', cbar=False)
plt.savefig(f'img/Clinic_feature_corr.svg', bbox_inches = 'tight')

## 5.、Construct data

willsampleTraining dataXandsupervision informationyseparate out，and thenTraining dataperformsplit，general splitting principleis80%-20%

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp

n_classes = 2
train_data = data[(data['group'] == 'train')]
train_ids = ids.loc[list(train_data.index)]
train_data = train_data.reset_index()
train_data = train_data.drop('index', axis=1)
y_data = train_data[labels]
X_data = train_data.drop(labels + ['group'], axis=1)

test_data = data[data['group'] != 'train']
test_ids = ids.loc[list(test_data.index)]
test_data = test_data.reset_index()
test_data = test_data.drop('index', axis=1)
y_test_data = test_data[labels]
X_test_data = test_data.drop(labels + ['group'], axis=1)

y_all_data = data[labels]
X_all_data = data.drop(labels + ['group'], axis=1)

column_names = X_data.columns
print(f"train sample size：{X_data.shape}, validation sample size：{X_test_data.shape}")

## 6.、Model filtering

Based on filtered data, perform initial model selection。Whenpreviously mainlyusetoisOnekeyin

1. SVM，Support Vector Machine,Reference。
2. KNN，KKNN,Reference。
3. Decision Tree，Decision Tree,Reference。
4. Random Forests, Random Forest,Reference。
5. XGBoost, bostingmethod。Reference。
6. LightGBM, bostingmethod，Reference。

In [ ]:
model_names = get_param_in_cwd('ml_models') or ['SVM', 'KNN', 'RandomForest', 'ExtraTrees', 'XGBoost', 'LightGBM', 'MLP', 'LR']
models = okcomp.comp1.create_clf_model(model_names)
model_names = list(models.keys())

### Cross-validation

`n_trails`Specify the number of random trials，Each trial uses80%train, random20%Test，Find the best model, and the corresponding best data split。

Here thedata has not beenusepreviously`Lasso`Filtered outfeatureperformtrain, theoretically，Feature filteringonly has a certain effect on linear models，e.g.,`SVM`、`LR`，butishas little effect on tree models，e.g.,`DecisionTree`、`Random`thissome。thereforedefaultdo not filter。

```python
def get_bst_split(X_data: pd.DataFrame, y_data: pd.DataFrame,
            models: dict, test_size=0.2, metric_fn=accuracy_score, n_trails=10,
            cv: bool = False, shuffle: bool = False, metric_cut_off: float = None, random_state=None):
    """
    Search for the best data split。
    Args:
        X_data: Training data
        y_data: Supervised data
        models: modelName，DictType、
        test_size: test ratio
        metric_fn: function for evaluating model performance，defaultaccuracy，Optionalroc_auc_score。
        n_trails: Number of attempts to search for the best data split。
        cv: Whether cross-validation，Default is False，WhenTruetime，n_trailsfor cross-validationn_fold
        shuffle: iswhether to perform random shuffling
        metric_cut_off: Whenmetric_fnvalue reaches how muchtimeperform truncation。
        random_state: Random seed

    Returns: {'max_idx': max_idx, "max_model": max_model, "max_metric": max_metric, "results": results}

    """
```

**Note：Adopted here: 【Data curation】，If you want to be rigorous，Please modify`n_trails=1`。**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

# randomusen_trailsdata split iteration，find the best data split，andsaveinresultsin。
results = okcomp.comp1.get_bst_split(X_data, y_data, models, test_size=0.2, metric_fn=roc_auc_score, n_trails=5, cv=True, random_state=0)
_, (X_train_sel, X_test_sel, y_train_sel, y_test_sel) = results['results'][results['max_idx']]
X_train_sel, X_test_sel, y_train_sel, y_test_sel = X_data, X_test_data, y_data, y_test_data
trails, _ = zip(*results['results'])
cv_results = pd.DataFrame(trails, columns=model_names)
# Visualize the performance of each model under different data splits。
sns.boxplot(data=cv_results)
plt.ylabel('AUC %')
plt.xlabel('Model Nmae')
plt.savefig(f'img/Clinic_model_cv.svg', bbox_inches = 'tight')

### Model filtering

Use the best data split for subsequent modeling。

**Note**: In general, papersuseisrandomly split the data，but some papersuse【Deliberately】curated data split。

In [ ]:
targets = []
for l in labels:
    new_models = list(okcomp.comp1.create_clf_model(model_names).values())
    for m in new_models:
        m.fit(X_train_sel, y_train_sel[l])
        y_pred = m.predict(X_test_sel)
    targets.append(new_models)

## 7.、predictresult

* predictions，2D data，eachlabelcorrespondingeach model's prediction result。
* pred_scores，2D data，eachlabelcorrespondingeach model's predicted probability。

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from onekey_algo.custom.components.delong import calc_95_CI
from onekey_algo.custom.components.metrics import analysis_pred_binary

predictions = [[(model.predict(X_train_sel), model.predict(X_test_sel)) 
                for model in target] for label, target in zip(labels, targets)]
pred_scores = [[(model.predict_proba(X_train_sel), model.predict_proba(X_test_sel)) 
                for model in target] for label, target in zip(labels, targets)]

metric = []
pred_sel_idx = []
for label, prediction, scores in zip(labels, predictions, pred_scores):
    pred_sel_idx_label = []
    for mname, (train_pred, test_pred), (train_score, test_score) in zip(model_names, prediction, scores):
        # computeTraining setindex
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_train_sel[label], 
                                                                                              train_score[:, 1])
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"{label}-train"))
                 
        # compute validation set metrics
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_test_sel[label], 
                                                                                              test_score[:, 1])
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"{label}-test"))
        # computethrescorrespondingsel idx
        pred_sel_idx_label.append(np.logical_or(test_score[:, 0] >= thres, test_score[:, 1] >= thres))
    
    pred_sel_idx.append(pred_sel_idx_label)
metric = pd.DataFrame(metric, index=None, columns=['model_name', 'Accuracy', 'AUC', '95% CI',
                                                   'Sensitivity', 'Specificity', 
                                                   'PPV', 'NPV', 'Precision', 'Recall', 'F1',
                                                   'Threshold', 'Task'])
metric

### Plot curves

plot different modelsaccuracy bar charts and line curves。

In [ ]:
import seaborn as sns

plt.subplot(211)
sns.barplot(x='model_name', y='Accuracy', data=metric, hue='Task')
plt.subplot(212)
sns.lineplot(x='model_name', y='Accuracy', data=metric, hue='Task')
plt.savefig(f'img/Clinic_model_acc.svg', bbox_inches = 'tight')

## plotROC curve
determine the best model，andPlot curves。

```python
def draw_roc(y_test, y_score, title='ROC', labels=None):
```

`sel_model = ['SVM', 'KNN']`parameterismodels you want to plotcorrespondingparameter。

In [ ]:
sel_model = model_names

for sm in sel_model:
    if sm in model_names:
        sel_model_idx = model_names.index(sm)
    
        # Plot all ROC curves
        plt.figure(figsize=(8, 8))
        for pred_score, label in zip(pred_scores, labels):
            okcomp.comp1.draw_roc([np.array(y_train_sel[label]), np.array(y_test_sel[label])], 
                                  pred_score[sel_model_idx], 
                                  labels=['Train', 'Test'], title=f"Model: {sm}")
            plt.savefig(f'img/Clinic_model_{sm}_roc_label.svg', bbox_inches = 'tight')

### summarize all models

In [ ]:
sel_model = model_names

for pred_score, label in zip(pred_scores, labels):
    pred_test_scores = []
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            pred_test_scores.append(pred_score[sel_model_idx][1])
    okcomp.comp1.draw_roc([np.array(y_test_sel[label])] * len(pred_test_scores), 
                          pred_test_scores, 
                          labels=sel_model, title=f"Model AUC")
    plt.savefig(f'img/Clinic_model_roc.svg', bbox_inches = 'tight')

### DCAdecision curve

In [ ]:
from onekey_algo.custom.components.comp1 import plot_DCA

for pred_score, label in zip(pred_scores, labels):
    pred_test_scores = []
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            okcomp.comp1.plot_DCA(pred_score[sel_model_idx][1][:,1], np.array(y_test_sel[label]),
                                  title=f'Clinic Model {sm} DCA')
            plt.savefig(f'img/Clinic_model_{sm}_dca.svg', bbox_inches = 'tight')

## Plot confusion matrix

Plot confusion matrix，[confusion matrixexplain](https://baike.baidu.com/item/%E6%B7%B7%E6%B7%86%E7%9F%A9%E9%98%B5/10087822?fr=aladdin)
`sel_model = ['SVM', 'KNN']`parameterismodels you want to plotcorrespondingparameter。

If you need to modify the label toNamemapping，modify`class_mapping={1:'1', 0:'0'}`

In [ ]:
# set plotting parameters
sel_model = model_names
c_matrix = {}

for sm in sel_model:
    if sm in model_names:
        sel_model_idx = model_names.index(sm)
        for idx, label in enumerate(labels):
            cm = okcomp.comp1.calc_confusion_matrix(predictions[idx][sel_model_idx][-1], y_test_sel[label],
#                                                     sel_idx = pred_sel_idx[idx][sel_model_idx],
                                                    class_mapping={1:'1', 0:'0'}, num_classes=2)
            c_matrix[label] = cm
            plt.figure(figsize=(5, 4))
            plt.title(f'Clinic Model:{sm}')
            okcomp.comp1.draw_matrix(cm, norm=False, annot=True, cmap='Blues', fmt=".3g")
            plt.savefig(f'img/Clinic_model_{sm}_cm.svg', bbox_inches = 'tight')

In [ ]:
import os
import numpy as np

sel_model = model_names
os.makedirs('results', exist_ok=True)
for idx, label in enumerate(labels):
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            target = targets[idx][sel_model_idx]
            # predicttrain and test setsdata。
            train_indexes = np.reshape(np.array(train_ids), (-1, 1)).astype(str)
            test_indexes = np.reshape(np.array(test_ids), (-1, 1)).astype(str)
            y_train_pred_scores = target.predict_proba(X_train_sel)
            y_test_pred_scores = target.predict_proba(X_test_sel)
            columns = ['ID'] + [f"{label}-{i}"for i in range(y_test_pred_scores.shape[1])]
            # save predictionstrain and test setsresult
            result_train = pd.DataFrame(np.concatenate([train_indexes, y_train_pred_scores], axis=1), columns=columns)
            result_train.to_csv(f'./results/Clinic_{sm}_train.csv', index=False)
            result_test = pd.DataFrame(np.concatenate([test_indexes, y_test_pred_scores], axis=1), columns=columns)
            result_test.to_csv(f'./results/Clinic_{sm}_test.csv', index=False)